# Historical Renewable Energy Patterns — Denmark

This notebook analyses **when renewable electricity is historically most available** in your part of Denmark,
so you know the best recurring windows to schedule EV charging.

**Data source:** [Energi Data Service](https://www.energidataservice.dk/) — the official Danish TSO (Energinet) open API.  
**Metric used:** `CO2Emission` (g CO₂eq / kWh) — the lower this is, the greener the electricity.
It already accounts for cross-border imports/exports, making it the cleanest single indicator.

Denmark is split into two electricity price areas:
| Area | Region | Examples |
|------|--------|----------|
| **DK1** | West Denmark — Jutland peninsula + Funen | Aarhus, Aalborg, Odense, Esbjerg, Herning |
| **DK2** | East Denmark — Zealand, Copenhagen, Bornholm | Copenhagen, Roskilde, Næstved, Hillerød |

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import seaborn as sns
import calendar
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
print('Libraries loaded.')

## Configuration
**Change `YOUR_CITY` to your Danish city.** The notebook will automatically map it to the correct price area (DK1 or DK2).
If your city is not found, set `PRICE_AREA` manually.

In [ ]:
# ============================================================
#  USER SETTINGS  —  change these
# ============================================================
YOUR_CITY    = "Aarhus"   # Your Danish city
LOOKBACK_DAYS = 90        # Days of history to analyse (30–365)
# ============================================================

# City → price area lookup (normalised to ASCII)
CITY_AREA_MAP = {
    # DK1 — West Denmark (Jutland + Funen)
    "aarhus": "DK1", "arhus": "DK1",
    "aalborg": "DK1", "alborg": "DK1",
    "odense": "DK1",
    "esbjerg": "DK1",
    "horsens": "DK1",
    "randers": "DK1",
    "kolding": "DK1",
    "vejle": "DK1",
    "silkeborg": "DK1",
    "herning": "DK1",
    "viborg": "DK1",
    "sonderborg": "DK1",
    "holstebro": "DK1",
    "fredericia": "DK1",
    "hjoerring": "DK1",
    "frederikshavn": "DK1",
    "skive": "DK1",
    "thisted": "DK1",
    "ikast": "DK1",
    "ribe": "DK1",
    "billund": "DK1",
    "svendborg": "DK1",
    "middelfart": "DK1",
    "nyborg": "DK1",
    "faaborg": "DK1",
    "ringkobing": "DK1",
    "struer": "DK1",
    "haderslev": "DK1",
    "aabenraa": "DK1",
    "tonder": "DK1",
    "give": "DK1",
    # DK2 — East Denmark (Copenhagen / Zealand / Bornholm)
    "copenhagen": "DK2", "kobenhavn": "DK2", "cph": "DK2",
    "frederiksberg": "DK2",
    "roskilde": "DK2",
    "helsingor": "DK2", "elsinore": "DK2",
    "hillerod": "DK2",
    "naestved": "DK2",
    "koege": "DK2",
    "slagelse": "DK2",
    "holbaek": "DK2",
    "bornholm": "DK2", "roenne": "DK2",
    "ringsted": "DK2",
    "greve": "DK2",
    "ballerup": "DK2",
    "gladsaxe": "DK2",
    "lyngby": "DK2",
    "taastrup": "DK2",
    "allerod": "DK2",
    "kalundborg": "DK2",
    "soroe": "DK2",
    "vordingborg": "DK2",
    "haslev": "DK2",
    "faxe": "DK2",
    "nykoebing falster": "DK2",
}

def normalise_city(name: str) -> str:
    """Lowercase + replace Danish special chars for lookup."""
    return (
        name.lower()
        .replace("ae", "ae").replace("oe", "oe").replace("aa", "aa")
        .replace("ae", "ae")  # ae ligature
        .replace("o", "o")
        .replace("ae", "ae").replace("oe", "oe").replace("aa", "aa")
        .encode("ascii", "ignore").decode()
        .strip()
    )

city_key = normalise_city(YOUR_CITY)

if city_key in CITY_AREA_MAP:
    PRICE_AREA = CITY_AREA_MAP[city_key]
else:
    # Fallback: manual override
    PRICE_AREA = "DK1"   # <-- change to "DK2" if you're east of the Great Belt
    print(f"WARNING: '{YOUR_CITY}' not found in lookup. Defaulting to {PRICE_AREA}.")
    print("Set PRICE_AREA = 'DK2' if you live in Copenhagen / Zealand / Bornholm.\n")

area_desc = (
    "West Denmark — Jutland peninsula + Funen"
    if PRICE_AREA == "DK1"
    else "East Denmark — Copenhagen / Zealand / Bornholm"
)
print(f"City       : {YOUR_CITY}")
print(f"Price area : {PRICE_AREA} ({area_desc})")
print(f"Lookback   : {LOOKBACK_DAYS} days")

## Fetch CO₂ Emission Data
We pull from the `CO2Emis` dataset — 5-minute resolution, already accounting for imports from Norway, Sweden, Germany, etc.

In [ ]:
def fetch_co2_data(price_area: str, start_dt: datetime, end_dt: datetime) -> pd.DataFrame:
    """
    Fetch CO2Emis data from Energi Data Service with automatic pagination.
    Returns a DataFrame with columns: Minutes5DK, CO2Emission.
    """
    url = "https://api.energidataservice.dk/dataset/CO2Emis"
    batch = 10_000
    offset = 0
    all_records = []

    while True:
        params = {
            "start": start_dt.strftime("%Y-%m-%dT%H:%M"),
            "end":   end_dt.strftime("%Y-%m-%dT%H:%M"),
            "filter": f'{{"PriceArea":["{price_area}"]}',
            "sort": "Minutes5UTC asc",
            "limit": batch,
            "offset": offset,
            "timezone": "dk",
        }
        resp = requests.get(url, params=params, timeout=60)
        resp.raise_for_status()
        records = resp.json().get("records", [])
        if not records:
            break
        all_records.extend(records)
        if len(records) < batch:
            break
        offset += batch

    return pd.DataFrame(all_records)


now_utc   = datetime.utcnow()
start_utc = now_utc - timedelta(days=LOOKBACK_DAYS)

print(f"Fetching {LOOKBACK_DAYS} days of 5-min CO2 data for {PRICE_AREA} ...")
df_raw = fetch_co2_data(PRICE_AREA, start_utc, now_utc)
print(f"Records received : {len(df_raw):,}")
print(f"Columns          : {list(df_raw.columns)}")
df_raw.head(3)

In [ ]:
# ── Clean & enrich ────────────────────────────────────────────────────────────
df = df_raw.copy()

# Prefer Danish local time column (Minutes5DK); fall back to UTC
time_col = "Minutes5DK" if "Minutes5DK" in df.columns else "Minutes5UTC"
df["ts"] = pd.to_datetime(df[time_col])
df = df.dropna(subset=["CO2Emission"]).sort_values("ts").reset_index(drop=True)

# Time features
df["hour"]    = df["ts"].dt.hour
df["weekday"] = df["ts"].dt.dayofweek       # 0 = Monday … 6 = Sunday
df["month"]   = df["ts"].dt.month
df["week_name"] = df["ts"].dt.day_name()
df["month_name"] = df["ts"].dt.strftime("%B")

# Renewable-greenness score: 0 = dirtiest, 100 = cleanest
co2_min, co2_max = df["CO2Emission"].min(), df["CO2Emission"].max()
df["green_score"] = 100 * (1 - (df["CO2Emission"] - co2_min) / (co2_max - co2_min + 1e-9))

print(f"Period   : {df['ts'].min().date()} → {df['ts'].max().date()}")
print(f"CO2 range: {co2_min:.0f} – {co2_max:.0f} g CO2/kWh")
print(f"Median   : {df['CO2Emission'].median():.0f} g CO2/kWh")
df[["ts", "CO2Emission", "green_score"]].head()

## Pattern 1 — Best Hours of the Day
Average CO₂ emission per hour of day.  
Lower bar = greener electricity = better time to charge.

In [ ]:
hourly = (
    df.groupby("hour")["CO2Emission"]
    .mean()
    .reset_index()
    .rename(columns={"CO2Emission": "mean_co2"})
)

# Colour bars by greenness
norm   = plt.Normalize(hourly["mean_co2"].min(), hourly["mean_co2"].max())
# Green = low CO2, red = high CO2 (reversed RdYlGn)
colors = plt.cm.RdYlGn_r(norm(hourly["mean_co2"]))

fig, ax = plt.subplots(figsize=(14, 4))
bars = ax.bar(hourly["hour"], hourly["mean_co2"], color=colors, edgecolor="white", linewidth=0.5)

ax.set_xlabel("Hour of day (Danish local time)")
ax.set_ylabel("Mean CO₂ (g/kWh)")
ax.set_title(f"{PRICE_AREA} — Average CO₂ Emission by Hour of Day  (lower = greener)")
ax.set_xticks(range(24))
ax.set_xticklabels([f"{h:02d}:00" for h in range(24)], rotation=45, ha="right", fontsize=9)

# Annotate best 3 hours
best_hours = hourly.nsmallest(3, "mean_co2")["hour"].tolist()
for h in best_hours:
    val = hourly.loc[hourly["hour"] == h, "mean_co2"].values[0]
    ax.annotate(f"{val:.0f}", xy=(h, val), xytext=(0, 6),
                textcoords="offset points", ha="center", fontsize=8,
                color="darkgreen", fontweight="bold")

# Add colourbar legend
sm = plt.cm.ScalarMappable(cmap="RdYlGn_r", norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=ax, label="g CO₂/kWh", shrink=0.6, pad=0.01)
plt.tight_layout()
plt.show()

print("\nTop-5 greenest hours (lowest average CO2):")
print(hourly.nsmallest(5, "mean_co2").to_string(index=False))

## Pattern 2 — Best Days of the Week
Average CO₂ per weekday.  
Wind tends to be more utilised on weekends when industrial consumption drops.

In [ ]:
WEEKDAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

daily = (
    df.groupby("week_name")["CO2Emission"]
    .mean()
    .reindex(WEEKDAY_ORDER)
    .reset_index()
    .rename(columns={"CO2Emission": "mean_co2"})
)

norm   = plt.Normalize(daily["mean_co2"].min(), daily["mean_co2"].max())
colors = plt.cm.RdYlGn_r(norm(daily["mean_co2"]))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(daily["week_name"], daily["mean_co2"], color=colors, edgecolor="white", linewidth=0.5)
ax.set_xlabel("Day of week")
ax.set_ylabel("Mean CO₂ (g/kWh)")
ax.set_title(f"{PRICE_AREA} — Average CO₂ Emission by Weekday  (lower = greener)")
ax.set_xticks(range(7))
ax.set_xticklabels(WEEKDAY_ORDER, rotation=15, ha="right")

sm = plt.cm.ScalarMappable(cmap="RdYlGn_r", norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=ax, label="g CO₂/kWh", shrink=0.6, pad=0.01)
plt.tight_layout()
plt.show()

best_day = daily.loc[daily["mean_co2"].idxmin(), "week_name"]
worst_day = daily.loc[daily["mean_co2"].idxmax(), "week_name"]
print(f"Greenest day  : {best_day}")
print(f"Dirtiest day  : {worst_day}")

## Pattern 3 — Seasonal Variation by Month
Summer = more solar (midday peaks), Winter = more wind (consistent but weather-dependent).

In [ ]:
monthly = (
    df.groupby(["month", "month_name"])["CO2Emission"]
    .agg(["mean", "std"])
    .reset_index()
    .sort_values("month")
)

month_labels = [calendar.month_abbr[m] for m in monthly["month"]]
norm   = plt.Normalize(monthly["mean"].min(), monthly["mean"].max())
colors = plt.cm.RdYlGn_r(norm(monthly["mean"]))

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(month_labels, monthly["mean"], color=colors,
              edgecolor="white", linewidth=0.5, label="Mean CO₂")
ax.errorbar(month_labels, monthly["mean"], yerr=monthly["std"],
            fmt="none", color="grey", capsize=4, linewidth=1, label="±1 std dev")

ax.set_xlabel("Month")
ax.set_ylabel("CO₂ (g/kWh)")
ax.set_title(f"{PRICE_AREA} — Average CO₂ Emission by Month  (lower = greener)")
ax.legend()
sm = plt.cm.ScalarMappable(cmap="RdYlGn_r", norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=ax, label="g CO₂/kWh", shrink=0.6, pad=0.01)
plt.tight_layout()
plt.show()

note = monthly.loc[monthly["mean"].idxmin(), "month_name"]
print(f"Historically greenest month in dataset: {note}")

## Pattern 4 — Hour × Weekday Heatmap
The richest view: find the intersections that are consistently the greenest (darkest green cells).

In [ ]:
pivot = (
    df.groupby(["weekday", "hour"])["CO2Emission"]
    .mean()
    .unstack(level="hour")
)
pivot.index = [WEEKDAY_ORDER[i] for i in pivot.index]

fig, ax = plt.subplots(figsize=(16, 4))
sns.heatmap(
    pivot,
    ax=ax,
    cmap="RdYlGn_r",          # red = dirty, green = clean (inverted because low CO2 = good)
    cmap_kws={"label": "g CO₂/kWh"},
    fmt=".0f",
    annot=True,
    annot_kws={"size": 7},
    linewidths=0.3,
    linecolor="white",
)
ax.set_title(f"{PRICE_AREA} — Mean CO₂ Emission by Hour × Weekday  (green = greenest electricity)")
ax.set_xlabel("Hour of day (Danish local time)")
ax.set_ylabel("")
ax.set_xticklabels([f"{h:02d}" for h in range(24)], rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

## Summary — Best Recurring Charging Windows
Based on the full historical dataset, these are the **hour-of-day slots with the lowest average CO₂**
across all days in the data — your safest default charging schedule.

In [ ]:
# Aggregate by hour and compute statistics
summary = (
    df.groupby("hour")["CO2Emission"]
    .agg(mean_co2="mean", median_co2="median", pct25="quantile",
         std_co2="std", count="count")
    .reset_index()
)
summary["green_score"] = (
    100 * (1 - (summary["mean_co2"] - summary["mean_co2"].min())
           / (summary["mean_co2"].max() - summary["mean_co2"].min() + 1e-9))
).round(1)

top5 = summary.nsmallest(5, "mean_co2").sort_values("hour")

print("=" * 60)
print(f"  TOP 5 CHARGING HOURS — {PRICE_AREA} ({YOUR_CITY})")
print("=" * 60)
for _, row in top5.iterrows():
    bar = "#" * int(row["green_score"] / 5)
    print(f"  {row['hour']:02d}:00 – {(row['hour']+1) % 24:02d}:00  "
          f"avg {row['mean_co2']:.0f} g CO2/kWh  "
          f"score {row['green_score']:.0f}/100  [{bar}]")
print("=" * 60)

# Charging advice by time-of-day
best_hour = int(summary.loc[summary["mean_co2"].idxmin(), "hour"])
print(f"\nBest single hour  : {best_hour:02d}:00 local Danish time")
print(f"Worst single hour : {int(summary.loc[summary['mean_co2'].idxmax(), 'hour']):02d}:00 local Danish time")

# Suggest a 3-4 hour charging window
rolling_mean = [
    (h, summary.set_index("hour")["mean_co2"].reindex(
        [(h + i) % 24 for i in range(4)]).mean())
    for h in range(24)
]
best_window_start = min(rolling_mean, key=lambda x: x[1])[0]
best_window_end   = (best_window_start + 4) % 24
print(f"\nBest 4-hour window: {best_window_start:02d}:00 – {best_window_end:02d}:00 local Danish time")
print("(Set your car's smart-charging timer to this window as a default.)")

## CO₂ Timeline — Last 30 Days
A raw look at how the CO₂ intensity has varied recently.

In [ ]:
last30 = df[df["ts"] >= df["ts"].max() - pd.Timedelta(days=30)].copy()

fig, ax = plt.subplots(figsize=(16, 4))

# Colour the fill by CO2 level
norm = plt.Normalize(last30["CO2Emission"].min(), last30["CO2Emission"].max())
cmap = plt.cm.RdYlGn_r

ax.plot(last30["ts"], last30["CO2Emission"], color="steelblue", linewidth=0.8, alpha=0.9)
ax.fill_between(last30["ts"], last30["CO2Emission"],
                alpha=0.2, color="steelblue")

# Shade hours in best-window green
in_window = last30[
    (last30["hour"] >= best_window_start) &
    (last30["hour"] < best_window_end)
]
ax.fill_between(in_window["ts"], 0, in_window["CO2Emission"],
                alpha=0.35, color="green", label=f"Best window ({best_window_start:02d}–{best_window_end:02d}h)")

ax.axhline(last30["CO2Emission"].mean(), color="orange", linestyle="--",
           linewidth=1.2, label=f"30-day mean ({last30['CO2Emission'].mean():.0f} g/kWh)")
ax.set_ylabel("CO₂ (g/kWh)")
ax.set_title(f"{PRICE_AREA} — CO₂ Intensity Last 30 Days  (green shading = historically best hours)")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()